## m steering

In [1]:
import glob
import numpy as np
from scipy.linalg import fractional_matrix_power
import time

def read_vectors_glob(pattern: str) -> np.array:
    results = []
    for path in glob.iglob(pattern):
        results.extend(np.load(path, allow_pickle=True))
    return results

In [2]:
neg_vectors = read_vectors_glob('../hidden_states/pos_vectors_horse_*.npy')
# pos_vectors = read_vectors_glob('../hidden_states/pos_vectors_motorcycle_*.npy')

In [ ]:
len(neg_vectors)

In [21]:
result = {}
for denoising_step in pos_vectors[0].keys():
    result[denoising_step] = {}
    for block in pos_vectors[0][denoising_step].keys():
        result[denoising_step][block] = []
        for layer in range(len(pos_vectors[0][denoising_step][block])):
            print(f'Processing step={denoising_step}, block={block}, layer={layer}')
            pos = np.stack([vector[denoising_step][block][layer] for vector in pos_vectors]).astype(np.float64)
            neg = np.stack([vector[denoising_step][block][layer] for vector in neg_vectors]).astype(np.float64)
            steering_vector = np.mean(pos, axis=0) - np.mean(neg, axis=0)
            print(np.linalg.norm(steering_vector))
            steering_vector /= np.linalg.norm(steering_vector)
            result[denoising_step][block].append(steering_vector.astype(np.float32))

Processing step=0, block=down, layer=0
0.8463489921532559
Processing step=0, block=down, layer=1
0.1621948743696451
Processing step=0, block=down, layer=2
0.10432178063395144
Processing step=0, block=down, layer=3
0.2047429447471583
Processing step=0, block=down, layer=4
2.4517617720495575
Processing step=0, block=down, layer=5
5.789367879876272
Processing step=0, block=down, layer=6
2.901707969644542
Processing step=0, block=down, layer=7
3.645370357342511
Processing step=0, block=down, layer=8
1.4228674703876638
Processing step=0, block=down, layer=9
0.45924506090227246
Processing step=0, block=down, layer=10
0.2746305385361668
Processing step=0, block=down, layer=11
0.4036199030099248
Processing step=0, block=down, layer=12
0.29216463475724525
Processing step=0, block=down, layer=13
0.27570505083063707
Processing step=0, block=down, layer=14
5.171917314277964
Processing step=0, block=down, layer=15
12.452278173198932
Processing step=0, block=down, layer=16
10.366980634802694
Process

In [22]:
import pickle

with open('../steering_vectors/horse_to_motorcycle_laion_steering_vectors_normed.pickle', 'wb') as fout:
    pickle.dump(result, fout)

# cov

In [2]:
vectors = read_vectors_glob('../hidden_states/pos_vectors_horse_*.npy')

In [4]:
result = {}
for denoising_step in vectors[0].keys():
    result[denoising_step] = {}
    for block in vectors[0][denoising_step].keys():
        result[denoising_step][block] = []
        for layer in range(len(vectors[0][denoising_step][block])):
            print(f'Processing step={denoising_step}, block={block}, layer={layer}')
            start = time.time()
            vec = np.stack([vector[denoising_step][block][layer] for vector in vectors]).astype(np.float64)
            vec /= np.linalg.norm(vec, axis=1, keepdims=True)

            mu = np.mean(vec, axis=0)
            sigma = np.dot(vec.T, vec) / (vec.shape[0] - 1)
            sigma -= np.outer(mu, mu)
            print(f'Took: {time.time() - start:.2f} s')
            
            result[denoising_step][block].append(sigma.astype(np.float32))

Processing step=0, block=down, layer=0
Took: 0.17 s
Processing step=0, block=down, layer=1
Took: 0.11 s
Processing step=0, block=down, layer=2
Took: 0.11 s
Processing step=0, block=down, layer=3
Took: 0.11 s
Processing step=0, block=down, layer=4
Took: 0.30 s
Processing step=0, block=down, layer=5
Took: 0.25 s
Processing step=0, block=down, layer=6
Took: 0.23 s
Processing step=0, block=down, layer=7
Took: 0.23 s
Processing step=0, block=down, layer=8
Took: 0.23 s
Processing step=0, block=down, layer=9
Took: 0.22 s
Processing step=0, block=down, layer=10
Took: 0.22 s
Processing step=0, block=down, layer=11
Took: 0.22 s
Processing step=0, block=down, layer=12
Took: 0.22 s
Processing step=0, block=down, layer=13
Took: 0.23 s
Processing step=0, block=down, layer=14
Took: 0.23 s
Processing step=0, block=down, layer=15
Took: 0.23 s
Processing step=0, block=down, layer=16
Took: 0.22 s
Processing step=0, block=down, layer=17
Took: 0.22 s
Processing step=0, block=down, layer=18
Took: 0.23 s
Pro

In [5]:
import pickle

with open('../horse_cov.pickle', 'wb') as fout:
    pickle.dump(result, fout)

## mm steering

In [23]:
def fractional_matrix_power_cov(A: np.ndarray, p: float, eps=1e-10):
    evals, evecs = np.linalg.eigh(A)
    evals = np.maximum(evals, 0)
    mask = (evals >= eps)
    evals = evals[mask]
    evecs = evecs[:, mask]
    return evecs @ np.diag(evals ** p) @ evecs.T

In [24]:
# pos_vectors, neg_vectors = neg_vectors, pos_vectors

In [25]:
result = {}
for denoising_step in pos_vectors[0].keys():
    result[denoising_step] = {}
    for block in pos_vectors[0][denoising_step].keys():
        result[denoising_step][block] = []
        for layer in range(len(pos_vectors[0][denoising_step][block])):
            print(f'Processing step={denoising_step}, block={block}, layer={layer}')
            start = time.time()
            pos = np.stack([vector[denoising_step][block][layer] for vector in pos_vectors]).astype(np.float64)
            # pos /= np.linalg.norm(pos, axis=1, keepdims=True)

            neg = np.stack([vector[denoising_step][block][layer] for vector in neg_vectors]).astype(np.float64)
            # neg /= np.linalg.norm(neg, axis=1, keepdims=True)

            mu_pos = np.mean(pos, axis=0)
            sigma_pos = np.dot(pos.T, pos) / (pos.shape[0] - 1)
            sigma_pos -= np.outer(mu_pos, mu_pos)

            mu_neg = np.mean(neg, axis=0)
            sigma_neg = np.dot(neg.T, neg) / (neg.shape[0] - 1)
            sigma_neg -= np.outer(mu_neg, mu_neg)

            
            sigma_neg_half = fractional_matrix_power_cov(sigma_neg, 0.5)
            sigma_neg_minus_half = fractional_matrix_power_cov(sigma_neg, -0.5)
            W = fractional_matrix_power_cov(sigma_neg_half @ sigma_pos @ sigma_neg_half, 0.5)
            W = sigma_neg_minus_half @ W @ sigma_neg_minus_half

            b = - W @ mu_neg + mu_pos

            print(f'Took: {time.time() - start:.2f} s')

            if W.dtype == np.complex128:
                print(f'Got unexpected complex values for step={denoising_step}, block={block}, layer={layer}, truncating...')
                W = np.real(W)
                b = np.real(b)
            
            result[denoising_step][block].append((W.astype(np.float32), b.astype(np.float32)))

Processing step=0, block=down, layer=0
Took: 0.33 s
Processing step=0, block=down, layer=1
Took: 0.28 s
Processing step=0, block=down, layer=2
Took: 0.28 s
Processing step=0, block=down, layer=3
Took: 0.28 s
Processing step=0, block=down, layer=4
Took: 1.04 s
Processing step=0, block=down, layer=5
Took: 0.94 s
Processing step=0, block=down, layer=6
Took: 0.93 s
Processing step=0, block=down, layer=7
Took: 0.93 s
Processing step=0, block=down, layer=8
Took: 0.94 s
Processing step=0, block=down, layer=9
Took: 0.94 s
Processing step=0, block=down, layer=10
Took: 0.96 s
Processing step=0, block=down, layer=11
Took: 0.94 s
Processing step=0, block=down, layer=12
Took: 0.94 s
Processing step=0, block=down, layer=13
Took: 0.96 s
Processing step=0, block=down, layer=14
Took: 0.93 s
Processing step=0, block=down, layer=15
Took: 0.93 s
Processing step=0, block=down, layer=16
Took: 0.93 s
Processing step=0, block=down, layer=17
Took: 0.94 s
Processing step=0, block=down, layer=18
Took: 0.94 s
Pro

In [26]:
result[0]['mid'][5][0]

array([[ 0.7864588 ,  0.0101397 ,  0.01199468, ..., -0.00631039,
        -0.02602464,  0.02117394],
       [ 0.0101397 ,  0.792389  ,  0.00188746, ...,  0.01337067,
        -0.02295238,  0.01678473],
       [ 0.01199468,  0.00188746,  0.7837137 , ...,  0.00908979,
         0.01465086, -0.00564717],
       ...,
       [-0.00631039,  0.01337067,  0.00908979, ...,  0.7982199 ,
        -0.00380174, -0.01305505],
       [-0.02602464, -0.02295238,  0.01465086, ..., -0.00380174,
         0.7814113 , -0.00236407],
       [ 0.02117394,  0.01678473, -0.00564717, ..., -0.01305505,
        -0.00236407,  0.7481372 ]], shape=(1280, 1280), dtype=float32)

In [304]:
for layer in ['down', 'mid', 'up']:
    for idx in range(len(result2[0][layer])):
        W, b = result[0][layer][idx]
        print(f'Layer {layer:4}, {idx:2}: |W|_2 = {np.linalg.norm(W, ord=2):.3f}, |b|_2 = {np.linalg.norm(b):.3f}')
        # print(np.linalg.svdvals(W)[:10])

Layer down,  0: |W|_2 = 3.415, |b|_2 = 0.672
Layer down,  1: |W|_2 = 3.089, |b|_2 = 0.163
Layer down,  2: |W|_2 = 2.858, |b|_2 = 0.129
Layer down,  3: |W|_2 = 3.135, |b|_2 = 0.265
Layer down,  4: |W|_2 = 5.893, |b|_2 = 2.081
Layer down,  5: |W|_2 = 7.296, |b|_2 = 5.868
Layer down,  6: |W|_2 = 7.292, |b|_2 = 3.300
Layer down,  7: |W|_2 = 6.730, |b|_2 = 2.701
Layer down,  8: |W|_2 = 6.904, |b|_2 = 1.368
Layer down,  9: |W|_2 = 5.509, |b|_2 = 0.584
Layer down, 10: |W|_2 = 4.878, |b|_2 = 0.312
Layer down, 11: |W|_2 = 4.699, |b|_2 = 0.300
Layer down, 12: |W|_2 = 4.618, |b|_2 = 0.321
Layer down, 13: |W|_2 = 4.609, |b|_2 = 0.867
Layer down, 14: |W|_2 = 6.601, |b|_2 = 5.933
Layer down, 15: |W|_2 = 7.276, |b|_2 = 13.119
Layer down, 16: |W|_2 = 7.483, |b|_2 = 10.802
Layer down, 17: |W|_2 = 7.496, |b|_2 = 5.679
Layer down, 18: |W|_2 = 7.978, |b|_2 = 4.107
Layer down, 19: |W|_2 = 7.090, |b|_2 = 3.995
Layer down, 20: |W|_2 = 7.035, |b|_2 = 3.086
Layer down, 21: |W|_2 = 7.221, |b|_2 = 3.092
Layer do

In [27]:
import pickle

with open('../steering_vectors/horse_to_motorcycle_laion_mm_steering_vectors_unnormed.pickle', 'wb') as fout:
    pickle.dump(result, fout)

# TODO:
- [x] Forward CASteer no norm
- [x] Inverse CASteer (normalized) with beta = 1
- [ ] CUDA libsvd NANs
- [ ] Horse to Motorcycle interpolation (forward CASteer with normalized, mmsteer unnormalized)

In [ ]:
|a| |b| cos (a, b)

In [28]:
pos_vectors, neg_vectors = neg_vectors, pos_vectors

In [29]:
result = {}
for denoising_step in pos_vectors[0].keys():
    result[denoising_step] = {}
    for block in pos_vectors[0][denoising_step].keys():
        result[denoising_step][block] = []
        for layer in range(len(pos_vectors[0][denoising_step][block])):
            print(f'Processing step={denoising_step}, block={block}, layer={layer}')
            start = time.time()
            pos = np.stack([vector[denoising_step][block][layer] for vector in pos_vectors]).astype(np.float64)
            # pos /= np.linalg.norm(pos, axis=1, keepdims=True)

            neg = np.stack([vector[denoising_step][block][layer] for vector in neg_vectors]).astype(np.float64)
            # neg /= np.linalg.norm(neg, axis=1, keepdims=True)

            mu_pos = np.mean(pos, axis=0)
            sigma_pos = np.dot(pos.T, pos) / (pos.shape[0] - 1)
            sigma_pos -= np.outer(mu_pos, mu_pos)

            mu_neg = np.mean(neg, axis=0)
            sigma_neg = np.dot(neg.T, neg) / (neg.shape[0] - 1)
            sigma_neg -= np.outer(mu_neg, mu_neg)

            
            sigma_neg_half = fractional_matrix_power_cov(sigma_neg, 0.5)
            sigma_neg_minus_half = fractional_matrix_power_cov(sigma_neg, -0.5)
            W = fractional_matrix_power_cov(sigma_neg_half @ sigma_pos @ sigma_neg_half, 0.5)
            W = sigma_neg_minus_half @ W @ sigma_neg_minus_half

            b = - W @ mu_neg + mu_pos

            print(f'Took: {time.time() - start:.2f} s')

            if W.dtype == np.complex128:
                print(f'Got unexpected complex values for step={denoising_step}, block={block}, layer={layer}, truncating...')
                W = np.real(W)
                b = np.real(b)
            
            result[denoising_step][block].append((W.astype(np.float32), b.astype(np.float32)))

Processing step=0, block=down, layer=0
Took: 0.40 s
Processing step=0, block=down, layer=1
Took: 0.33 s
Processing step=0, block=down, layer=2
Took: 0.29 s
Processing step=0, block=down, layer=3
Took: 0.28 s
Processing step=0, block=down, layer=4
Took: 0.97 s
Processing step=0, block=down, layer=5
Took: 1.05 s
Processing step=0, block=down, layer=6
Took: 0.95 s
Processing step=0, block=down, layer=7
Took: 0.97 s
Processing step=0, block=down, layer=8
Took: 1.01 s
Processing step=0, block=down, layer=9
Took: 0.97 s
Processing step=0, block=down, layer=10
Took: 0.95 s
Processing step=0, block=down, layer=11
Took: 0.95 s
Processing step=0, block=down, layer=12
Took: 0.96 s
Processing step=0, block=down, layer=13
Took: 0.95 s
Processing step=0, block=down, layer=14
Took: 0.95 s
Processing step=0, block=down, layer=15
Took: 0.96 s
Processing step=0, block=down, layer=16
Took: 0.94 s
Processing step=0, block=down, layer=17
Took: 0.94 s
Processing step=0, block=down, layer=18
Took: 0.95 s
Pro

In [30]:
import pickle

with open('../steering_vectors/motorcycle_to_horse_laion_mm_steering_vectors_unnormed.pickle', 'wb') as fout:
    pickle.dump(result, fout)

## LEACE

In [2]:
import sys
sys.path.append('/Users/astepanov/repos/mmsteer/')

In [3]:
import torch
from utils import unpickle, convert_to_widest_dtype, fractional_matrix_power_cov_torch

/Users/astepanov/repos/mmsteer/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/astepanov/repos/mmsteer/.venv/lib/python3.13/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'


In [4]:
device = torch.device('mps')

In [13]:
ALL_COVARIANCES_PATH = "../ckpt/llama2-7b-chat_alpaca_all_self_attn_all/pos_covariances_1000.pickle"
ALL_MEANS_PATH       = "../ckpt/llama2-7b-chat_alpaca_all_self_attn_all/pos_means_1000.pickle"
NEG_MEANS_PATH       = "../ckpt/llama2-7b-chat_alpaca_toxicity_self_attn_last/neg_means_1000.pickle"
POS_MEANS_PATH       = "../ckpt/llama2-7b-chat_alpaca_toxicity_self_attn_last/pos_means_1000.pickle"

In [14]:
all_covariances = unpickle(ALL_COVARIANCES_PATH)
all_means = unpickle(ALL_MEANS_PATH)
neg_means = unpickle(NEG_MEANS_PATH)
pos_means = unpickle(POS_MEANS_PATH)

In [15]:
force_double = False

In [20]:
for layer_idx in range(32):
    indexer = lambda x: x[0]['LLM'][layer_idx]
    sigma = convert_to_widest_dtype(indexer(all_covariances), device=device, force_double=force_double)
    m_neutral = convert_to_widest_dtype(indexer(all_means), device=device, force_double=force_double)
    m_neg = convert_to_widest_dtype(indexer(neg_means), device=device, force_double=force_double)
    m_pos = convert_to_widest_dtype(indexer(pos_means), device=device, force_double=force_double)
    cos = torch.dot(m_pos.squeeze(0), m_neg.squeeze(0)) / m_pos.squeeze(0).norm() / m_neg.squeeze(0).norm()
    print(f'{layer_idx}: {(m_pos - m_neg).norm(dim=-1)} {cos}')

0: tensor([0.1347], device='mps:0') 0.9934439063072205
1: tensor([0.2027], device='mps:0') 0.9885653257369995
2: tensor([0.1491], device='mps:0') 0.9864850044250488
3: tensor([0.2565], device='mps:0') 0.9892691373825073
4: tensor([0.2944], device='mps:0') 0.9894146919250488
5: tensor([0.5652], device='mps:0') 0.969781219959259
6: tensor([1.1785], device='mps:0') 0.92925626039505
7: tensor([0.9601], device='mps:0') 0.9211862087249756
8: tensor([1.9388], device='mps:0') 0.8396617770195007
9: tensor([5.0196], device='mps:0') 0.7138270735740662
10: tensor([4.7627], device='mps:0') 0.614035427570343
11: tensor([4.4069], device='mps:0') 0.5282272696495056
12: tensor([4.6148], device='mps:0') 0.5342133045196533
13: tensor([5.0774], device='mps:0') 0.5606318712234497
14: tensor([5.8952], device='mps:0') 0.4818580746650696
15: tensor([8.1899], device='mps:0') 0.42264774441719055
16: tensor([6.3076], device='mps:0') 0.4986703395843506
17: tensor([3.7688], device='mps:0') 0.5233935117721558
18: t

In [211]:
steering_vector = pos_means - neg_means

In [217]:
(neg_means - mean).norm()

TypeError: unsupported operand type(s) for -: 'dict' and 'Tensor'

In [218]:
(pos_means - mean).norm()

TypeError: unsupported operand type(s) for -: 'dict' and 'Tensor'

In [118]:
sigma_minus_half = fractional_matrix_power_cov_torch(sigma, -0.5, eps=1e-10)
sigma_plus_half = fractional_matrix_power_cov_torch(sigma, 0.5, eps=1e-10)

In [119]:
diff = (sigma_plus_half @ sigma_minus_half)[0] - torch.eye(4096, device=sigma_minus_half.device, dtype=sigma_minus_half.dtype)

In [120]:
assert diff.flatten().abs().max() <= 0.05

In [121]:
steering_proj = sigma_minus_half @ steering_vector.unsqueeze(-1)

In [122]:
P = sigma_plus_half @ (steering_proj @ torch.linalg.pinv(steering_proj)) @ sigma_minus_half

In [123]:
assert (P@P - P).flatten().abs().max() <= 1e-5

In [162]:
x = torch.randn(1, 4096, 1, dtype=steering_vector.dtype, device=steering_vector.device)

In [163]:
torch.dot(
    steering_vector.squeeze(0),
    (P @ x).squeeze(0, -1)
) / steering_vector.norm() / (P @ x).squeeze(0, -1).norm()

tensor(1.0000, device='mps:0')

In [146]:
(P @ (steering_vector + torch.tensor([1] * 4095 + [0.2523 / steering_vector[0, 4095]], dtype=torch.float32, device=device).unsqueeze(0)).unsqueeze(-1)).squeeze(0, -1)

tensor([-0.0830, -0.0717, -0.2964,  ..., -0.4659,  0.2655,  0.1492],
       device='mps:0')

In [147]:
steering_vector

tensor([[ 0.0039,  0.0033,  0.0138,  ...,  0.0217, -0.0124, -0.0069]],
       device='mps:0')